# Práctica 3 · Exploración conformacional automática
### Búsqueda conformacional y estructuras de baja energía

| Campo | Valor |
|---|---|
| **Bloque** | 1 — Generación de estructuras y espacio conformacional |
| **Nivel** | Básico |
| **Tiempo estimado** | 1.5 h (semilla) + 2 h (bosque y análisis) |
| **Pipeline** | Geometría GFN2-xTB (P1) → iMTD-GC (CREST) → ensemble → filtrado RMSD/ΔE → poblaciones Boltzmann → dataset |
| **Prerequisito** | Prácticas 1 y 2 completadas |
| **Hardware mínimo** | 4 GB RAM, 4 núcleos ≥ 2 GHz |

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/drive/p03_crest)


---
## Introducción química

Una molécula flexible no existe en una única geometría: explora continuamente
un paisaje de energía potencial (PES) poblando distintos confórmeros según la
distribución de Boltzmann. Calcular sólo el mínimo de energía equivale a
estudiar una fracción arbitraria de la población real.

**CREST** (*Conformer-Rotamer Ensemble Sampling Tool*, Grimme et al.) combina
GFN2-xTB con el algoritmo **iMTD-GC** (*iterative Metadynamics + Genetic Crossing*):

```
Geometría xTB  →  iMTD-GC (CREST)  →  Ensemble crudo
                      │                      │
              Metadinámica LT         Filtrado RMSD < 0.125 Å
              Cruce genético          Ventana ΔE < 25 kJ/mol
                                             │
                                    Poblaciones Boltzmann
                                    Dataset conformacional
```

**Importancia práctica:**
- *Síntesis asistida:* el confórmero de mínima energía determina la
  selectividad facial de una reacción.
- *Diseño de fármacos:* el confórmero bioactivo ≠ el más estable en solución;
  el ensemble completo permite estimar el costo entrópico de unión.
- *Espectroscopía computacional:* comparar correctamente con espectros
  experimentales requiere ponderar todos los confórmeros por sus poblaciones.

> **🌱 Semilla:** N-acetil-L-alanina metilamida (Ac-Ala-NHMe, `CC(=O)N[C@@H](C)C(=O)NC`) —
> el dipéptido modelo más estudiado, con benchmark vs MP2/6-311+G** (Guo et al. 2014).
>
> **🌳 Bosque:** 35 moléculas — alcanos lineales, dipéptidos modelo, fármacos, macrociclos.


---
## Marco teórico esencial

**1. PES y confórmeros.** Los confórmeros son mínimos locales en el espacio
de $3N-6$ dimensiones. La diferencia energética entre confórmeros es 1–20 kJ/mol,
comparable a $k_BT \approx 2.5$ kJ/mol a 298 K — múltiples confórmeros
coexisten con poblaciones significativas a temperatura ambiente.

**2. Metadinámica de baja temperatura.** Añade un potencial de sesgo gaussiano
acumulativo que penaliza regiones ya visitadas, forzando exploración de zonas
nuevas sin romper enlaces (temperatura virtual: 100–400 K).

**3. Cruce genético (GC).** Combina fragmentos de distintos confórmeros
(definidos por ángulos diedros) para generar nuevas estructuras — cobertura
más sistemática que la metadinámica sola.

**4. Filtrado.** RMSD < 0.125 Å para duplicados; $\Delta E < 25$ kJ/mol
sobre el mínimo global.

**5. Promedio de Boltzmann.**

$$\langle A \rangle = \sum_i p_i A_i, \qquad
p_i = \frac{e^{-G_i/RT}}{\sum_j e^{-G_j/RT}}$$

### Mínimos de referencia del mapa de Ramachandran (Ac-Ala-NHMe, MP2/6-311+G**)

| Mínimo | φ ref. | ψ ref. | Descripción |
|---|---|---|---|
| $C_7^{eq}$ | −82° | +65° | Puente H intramolecular de 7 miembros |
| $\alpha_R$ | −64° | −40° | Hélice alfa derecha |
| $C_5$ | −158° | +153° | Conformación extendida β |

**Criterio de aceptación:** desviación < 15° en φ y ψ respecto a la referencia.


---
## ⚙️ Configuración del entorno

Ejecuta esta celda primero. Verifica CREST, xTB y las librerías de análisis.
En Colab instala las dependencias automáticamente.


In [ ]:
import sys, subprocess, os, re, shutil, pathlib, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib; matplotlib.rcParams['figure.dpi'] = 120
warnings.filterwarnings('ignore')

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    subprocess.run(['pip', 'install', '-q', 'rdkit', 'py3Dmol', 'plotly'], check=False)

try:
    from rdkit import Chem
    from rdkit.Chem import AllChem, rdmolfiles, rdMolDescriptors
    RDKIT_OK = True; print('RDKit OK')
except ImportError:
    RDKIT_OK = False; print('RDKit no disponible')

try:
    import py3Dmol; PY3DMOL_OK = True; print('py3Dmol OK')
except ImportError:
    PY3DMOL_OK = False; print('py3Dmol no disponible')

try:
    import plotly.express as px
    import plotly.graph_objects as go
    PLOTLY_OK = True; print('Plotly OK')
except ImportError:
    PLOTLY_OK = False; print('Plotly no disponible')

CREST_OK = subprocess.run(['which','crest'], capture_output=True).returncode == 0
XTB_OK   = subprocess.run(['which','xtb'],   capture_output=True).returncode == 0
print('crest:', 'OK' if CREST_OK else 'no encontrado -- conda install -c conda-forge crest')
print('xtb:  ', 'OK' if XTB_OK   else 'no encontrado (incluido en CREST)')

# Constantes globales
R     = 8.314e-3  # kJ/(mol·K)
T_K   = 298.15    # K
EH2KJ = 2625.5    # kJ/mol por Hartree

SMILES_SEMILLA = 'CC(=O)N[C@@H](C)C(=O)NC'  # Ac-Ala-NHMe
NOMBRE         = 'Ac-Ala-NHMe'
print(f'Listo. R={R}, T={T_K} K')


---
## 📝 Preguntas previas

Responde **antes** de ejecutar cualquier celda de cálculo. Tus respuestas
quedan registradas para comparar al final con los resultados reales.


**Pregunta 1 (Conceptual).** El butano tiene mínimos *anti* y *gauche*.
A 298 K, ¿cuál es la fracción aproximada en conformación *anti* si
$\Delta G_{anti \to gauche} \approx 2.5$ kJ/mol?
(Hay dos *gauche* equivalentes — tenlo en cuenta.) ¿Cambiaría a 500 K?


In [ ]:
respuesta_1 = (
    'La fraccion anti a 298K es aproximadamente ... porque ...\n'
    'A 500K cambiaria/no cambiaria porque ...'
)
print('Respuesta 1 guardada.')


**Pregunta 2 (Predictivo).** La Ac-Ala-NHMe tiene 4 enlaces rotativos
relevantes. Si cada uno pudiera adoptar 3 posiciones (*g+*, *anti*, *g−*),
¿cuántos confórmeros teóricos existirían? ¿Por qué CREST encuentra muchos menos?


In [ ]:
respuesta_2 = (
    'Conformeros teoricos maximos: 3^4 = ...\n'
    'CREST encuentra menos porque ...'
)
print('Respuesta 2 guardada.')


**Pregunta 3 (Crítico).** La metadinámica de CREST corre con GFN2-xTB, no con
DFT. ¿Qué sesgo podría introducir el método semiempírico en la búsqueda
conformacional? ¿Cómo validarías que el ensemble encontrado es completo?


In [ ]:
respuesta_3 = (
    'El sesgo potencial de GFN2-xTB vs DFT en conformacional es ...\n'
    'Para validar la completitud del ensemble podria ...'
)
print('Respuesta 3 guardada.')


---
## 🌱 Semilla · Paso 1 — Preparar la geometría de entrada

CREST recibe una geometría 3D pre-optimizada. Generamos con ETKDG v3 + MMFF94
(igual que en P1) o usamos `p03_semilla_inicio.xyz` si está disponible.

**Molécula semilla:** Ac-Ala-NHMe — 4 enlaces rotativos relevantes:
φ y ψ del esqueleto peptídico y dos del grupo acetilo.

```
SMILES: CC(=O)N[C@@H](C)C(=O)NC

  CH3-CO-NH-Ca(CH3)-CO-NH-CH3
              ↑
          phi y psi: diedros del esqueleto peptidico
```


In [ ]:
XYZ_ENTRADA = 'p03_semilla_inicio.xyz'

if os.path.exists(XYZ_ENTRADA):
    print(f'Usando geometria provista: {XYZ_ENTRADA}')
    with open(XYZ_ENTRADA) as f:
        lineas = f.readlines()
    print(f'  Atomos: {lineas[0].strip()}  |  {lineas[1].strip()}')

elif RDKIT_OK:
    print('Generando geometria ETKDG v3 + MMFF94...')
    mol   = Chem.MolFromSmiles(SMILES_SEMILLA)
    mol_h = Chem.AddHs(mol)
    params = AllChem.ETKDGv3(); params.randomSeed = 42
    AllChem.EmbedMolecule(mol_h, params)
    ff = AllChem.MMFFGetMoleculeForceField(
             mol_h, AllChem.MMFFGetMoleculeProperties(mol_h))
    if ff: ff.Minimize(maxIts=2000)
    rdmolfiles.MolToXYZFile(mol_h, XYZ_ENTRADA)
    print(f'  Guardada: {XYZ_ENTRADA}  ({mol_h.GetNumAtoms()} atomos)')

    if XTB_OK:
        print('  Pre-optimizando con GFN2-xTB...')
        subprocess.run(['xtb', XYZ_ENTRADA, '--opt', 'normal',
                        '--chrg', '0', '--uhf', '0', '--gfn', '2'],
                       capture_output=True)
        if os.path.exists('xtbopt.xyz'):
            shutil.copy('xtbopt.xyz', XYZ_ENTRADA)
            print('  Geometria xTB copiada a', XYZ_ENTRADA)
else:
    print('Ni el archivo XYZ ni RDKit disponibles.')
    print('Descarga p03_semilla_inicio.xyz del repositorio del curso.')


---
## 🌱 Semilla · Paso 2 — Búsqueda conformacional con CREST (iMTD-GC)

### Comando y flags

```bash
crest p03_semilla_inicio.xyz --gfn2 --T 4 --ewin 25 --rthr 0.125
```

| Flag | Significado |
|---|---|
| `--gfn2` | Método de energía: GFN2-xTB |
| `--T 4` | Núcleos de CPU (ajustar según hardware) |
| `--ewin 25` | Ventana de energía en kJ/mol |
| `--rthr 0.125` | Umbral RMSD en Å para distinguir confórmeros |

### Archivos de salida

| Archivo | Contenido |
|---|---|
| `crest_conformers.xyz` | Ensemble final (XYZ multi-estructura) |
| `crest.energies` | Energías en Hartree, ordenadas |
| `crest_best.xyz` | Confórmero global |
| `crest_run.log` | Log completo con estadísticas del muestreo |

> ⚠️ **Tiempo de ejecución:** 5–20 min para Ac-Ala-NHMe con 4 núcleos.
> Si tarda > 45 min, interrumpe con Ctrl+C y analiza los confórmeros parciales.


In [ ]:
# Si CREST esta disponible, ejecutar la busqueda real.
# Si no, generamos un ensemble simulado basado en la literatura (Guo et al. 2014).

if CREST_OK and os.path.exists(XYZ_ENTRADA):
    print('Ejecutando CREST iMTD-GC... (5-20 min)')
    result = subprocess.run(
        ['crest', XYZ_ENTRADA, '--gfn2', '--T', '4',
         '--ewin', '25', '--rthr', '0.125'],
        capture_output=True, text=True
    )
    with open('crest_run.log', 'w') as f:
        f.write(result.stdout)
    ok = 'CREST terminated normally' in result.stdout
    print('CREST terminated normally:', ok)
    m = re.search(r'(\d+)\s+unique conformers', result.stdout)
    if m: print(f'Conformeros unicos: {m.group(1)}')

else:
    print('CREST no disponible -- generando ensemble simulado de referencia.')
    print('Valores basados en Guo et al. 2014 + calculos GFN2-xTB de referencia.')

    # 12 conformeros conocidos de la Ac-Ala-NHMe
    # (dE_kJ, phi_deg, psi_deg)
    conf_ref = [
        (0.000,  -82,  +65),   # C7eq  -- dominante en gas
        (2.100,  -64,  -40),   # alphaR
        (4.300, -158, +153),   # C5    -- extendido
        (6.800,  -80,  +80),
        (8.200,  +60,  +40),   # alphaL
        (9.500, -120, +120),
        (11.30,  -70,  -50),
        (13.10,  +80,  -70),
        (15.40,  -90,  +90),   # C7ax
        (17.20, +170,  +40),
        (19.80, -150,  -50),
        (22.50,  +50, -150),
    ]

    with open('crest.energies', 'w') as f:
        for i, (dE, phi, psi) in enumerate(conf_ref, 1):
            E_Eh = -35.450000 + dE / EH2KJ
            f.write(f'{i:4d}  {E_Eh:.6f}\n')

    with open('crest_run.log', 'w') as f:
        f.write('CREST (SIMULADO -- fallback pedagogico)\n')
        f.write('Molecula: Ac-Ala-NHMe\n')
        f.write('Rondas iMTD-GC ejecutadas: 4\n')
        f.write('12 unique conformers found.\n')
        f.write('CREST terminated normally.\n')

    # XYZ multiconformero minimalista
    n_at = 20
    syms = ['C','C','O','N','C','C','C','O','N','C',
            'H','H','H','H','H','H','H','H','H','H']
    np.random.seed(42)
    with open('crest_conformers.xyz', 'w') as f:
        for i, (dE, phi, psi) in enumerate(conf_ref, 1):
            f.write(f'{n_at}\n')
            f.write(f'CREST conf {i} dE={dE:.3f} kJ/mol phi={phi} psi={psi}\n')
            for j, sym in enumerate(syms):
                x = np.random.uniform(-3,3) + np.cos(np.radians(phi+j*18))
                y = np.random.uniform(-3,3) + np.sin(np.radians(psi+j*18))
                z = np.random.uniform(-2,2)
                f.write(f'{sym}  {x:.4f}  {y:.4f}  {z:.4f}\n')
    shutil.copy('crest_conformers.xyz', 'crest_best.xyz')
    print('  Ensemble simulado generado: 12 conformeros')

print('Paso 2 completado.')


---
## 🌱 Semilla · Paso 3 — Anatomía del log de CREST

El log registra el progreso del muestreo. Elementos clave a verificar:

- **Rondas iMTD-GC ejecutadas** — mínimo recomendado: 3.
- **Convergencia** — el número de confórmeros únicos no debe aumentar entre
  la penúltima y la última ronda.
- **`CREST terminated normally`** — confirma finalización correcta.


In [ ]:
with open('crest_run.log') as f:
    log = f.read()

print('=' * 55)
print('  EXTRACTO DEL LOG DE CREST')
print('=' * 55)

rondas = re.findall(r'[Rr]ound\s*(\d+)', log)
if rondas:
    print(f'  Rondas iMTD-GC: {max(int(r) for r in rondas)}')

m_conf = re.search(r'(\d+)\s+unique conformers', log)
if m_conf:
    print(f'  Conformeros unicos finales: {m_conf.group(1)}')

if 'CREST terminated normally' in log:
    print('  Finalizacion: OK (CREST terminated normally)')
else:
    print('  ATENCION: CREST no termino normalmente -- revisar log completo')

print()
print('Criterios de convergencia:')
print('  - Rondas iMTD-GC >= 3')
print('  - N conformeros estable entre penultima y ultima ronda')
print('  - CREST terminated normally')


---
## 🌱 Semilla · Paso 4 — Energías y distribución de Boltzmann

### Formato de `crest.energies`
```
   1  -35.450000   ← índice  E (Hartree)
   2  -35.449200
   ...
```
Conversión: 1 Hartree = 2625.5 kJ/mol.
Trabajamos con ΔE relativo al mínimo (numéricamente estable).


In [ ]:
energias = np.loadtxt('crest.energies', usecols=1)
n_conf   = len(energias)
print(f'Conformeros en el ensemble: {n_conf}')

dE_kJ = (energias - energias.min()) * EH2KJ
pesos = np.exp(-dE_kJ / (R * T_K))
pop   = pesos / pesos.sum() * 100

# phi/psi del ensemble simulado (o se calcularán en el Paso 5)
phi_ref = [-82,-64,-158,-80,+60,-120,-70,+80,-90,+170,-150,+50]
psi_ref = [+65,-40,+153,+80,+40,+120,-50,-70,+90,+40,-50,-150]

df_conf = pd.DataFrame({
    'conformero'   : range(1, n_conf+1),
    'E_Eh'         : energias,
    'dE_kJ_mol'    : dE_kJ.round(3),
    'poblacion_pct': pop.round(3),
    'phi'          : phi_ref[:n_conf],
    'psi'          : psi_ref[:n_conf],
})

print()
print('=== Distribucion de Boltzmann (298 K) ===')
print(df_conf[['conformero','dE_kJ_mol','poblacion_pct','phi','psi']]
      .head(10).round(2).to_string(index=False))

C1 = df_conf.iloc[0]
print(f'\nConformero dominante: C1  {C1.poblacion_pct:.1f}%'
      f'  phi={C1.phi:.0f}  psi={C1.psi:.0f}')

df_pop_sort = df_conf.sort_values('dE_kJ_mol')
n_90 = int((df_pop_sort['poblacion_pct'].cumsum() < 90).sum()) + 1
print(f'Conformeros para acumular el 90% de poblacion: {n_90}')

df_conf.to_csv('p03_ensemble.csv', index=False)
print('Guardado: p03_ensemble.csv')


---
## 🌱 Semilla · Paso 5 — Calcular ángulos diedros φ y ψ

**Definiciones para Ac-Ala-NHMe:**
- **φ** = diedro C(acetilo)–N–Cα–C(carbonilo)
- **ψ** = diedro N–Cα–C(carbonilo)–N(amida)

Para identificar los índices atómicos (base 0) en tu XYZ:
1. Abre `crest_best.xyz` en Avogadro2 o un editor de texto
2. Localiza los 5 átomos del esqueleto peptídico
3. Sus posiciones en el archivo son los índices

| Símbolo | Índice aprox. | Rol en diedro |
|---|---|---|
| C carbonilo acetilo | 1 | C_prev para φ |
| N del NH | 3 | N para φ |
| Cα | 4 | Cα para φ y ψ |
| C carbonilo peptídico | 6 | C para ψ |
| N de la amida | 8 | N_next para ψ |


In [ ]:
def leer_ensemble_xyz(archivo):
    estructuras = []
    try:
        with open(archivo) as f:
            lineas = f.readlines()
        i = 0
        while i < len(lineas):
            line = lineas[i].strip()
            if not line: i += 1; continue
            n = int(line)
            comentario = lineas[i+1].strip()
            atomos, coords = [], []
            for j in range(n):
                p = lineas[i+2+j].split()
                atomos.append(p[0])
                coords.append([float(x) for x in p[1:4]])
            estructuras.append((atomos, np.array(coords), comentario))
            i += n + 2
    except Exception as e:
        print(f'Error: {e}')
    return estructuras

def angulo_diedro(p1, p2, p3, p4):
    b1 = p2 - p1; b2 = p3 - p2; b3 = p4 - p3
    n1 = np.cross(b1, b2)
    n2 = np.cross(b2, b3)
    m1 = np.cross(n1, b2 / (np.linalg.norm(b2) + 1e-12))
    return np.degrees(np.arctan2(np.dot(m1, n2), np.dot(n1, n2)))

estructuras = leer_ensemble_xyz('crest_conformers.xyz')
print(f'Estructuras leidas: {len(estructuras)}')

# Indices para Ac-Ala-NHMe generada con RDKit (ajustar si difieren)
IDX = dict(C_prev=1, N=3, CA=4, C=6, N_next=8)

phi_calc, psi_calc = [], []
max_idx = max(IDX.values())
for atomos, coords, _ in estructuras:
    if len(coords) > max_idx:
        phi_calc.append(round(angulo_diedro(
            coords[IDX['C_prev']], coords[IDX['N']],
            coords[IDX['CA']],    coords[IDX['C']]), 1))
        psi_calc.append(round(angulo_diedro(
            coords[IDX['N']],     coords[IDX['CA']],
            coords[IDX['C']],     coords[IDX['N_next']]), 1))
    else:
        # Mantener los valores del ensemble simulado
        j = len(phi_calc)
        phi_calc.append(df_conf['phi'].iloc[j] if j < len(df_conf) else None)
        psi_calc.append(df_conf['psi'].iloc[j] if j < len(df_conf) else None)

n = min(len(df_conf), len(phi_calc))
df_conf.loc[:n-1, 'phi'] = phi_calc[:n]
df_conf.loc[:n-1, 'psi'] = psi_calc[:n]
df_conf.to_csv('p03_ensemble.csv', index=False)
print('phi/psi calculados y guardados en p03_ensemble.csv')
print(df_conf[['conformero','dE_kJ_mol','poblacion_pct','phi','psi']].round(1).to_string(index=False))


---
## 🌱 Semilla · Paso 6 — Validación vs mapa de Ramachandran de referencia

Comparamos los confórmeros del ensemble con los mínimos de MP2/6-311+G**
(Guo et al. 2014). El criterio de aceptación es desviación < 15° en φ y ψ.


In [ ]:
referencia_rama = {
    'C7eq'  : {'phi': -82, 'psi':  +65},
    'alphaR': {'phi': -64, 'psi':  -40},
    'C5'    : {'phi':-158, 'psi': +153},
}

df_dr = df_conf.dropna(subset=['phi','psi']).copy()

print('=== Validacion vs Ramachandran MP2 (Guo et al. 2014) ===')
print(f'  {"Minimo":<10} {"phi_ref":>8} {"phi_calc":>9} {"psi_ref":>8} {"psi_calc":>9} {"dPhi":>6} {"dPsi":>6}  OK?')
print('  ' + '-'*72)

todos_ok = True
for nombre, ref in referencia_rama.items():
    mejor, d_min = None, 9999
    for _, row in df_dr.iterrows():
        dphi = min(abs(row.phi - ref['phi']), 360 - abs(row.phi - ref['phi']))
        dpsi = min(abs(row.psi - ref['psi']), 360 - abs(row.psi - ref['psi']))
        d = (dphi**2 + dpsi**2)**0.5
        if d < d_min:
            d_min, mejor, dp, ds = d, row, dphi, dpsi
    ok = 'OK' if (dp < 15 and ds < 15) else 'FUERA'
    if ok != 'OK': todos_ok = False
    print(f'  {nombre:<10} {ref["phi"]:>8.0f} {mejor.phi:>9.1f}'
          f' {ref["psi"]:>8.0f} {mejor.psi:>9.1f}'
          f' {dp:>6.1f} {ds:>6.1f}  {ok}')

print()
if todos_ok:
    print('Validacion SUPERADA: los 3 minimos de referencia encontrados.')
else:
    print('Algunos minimos fuera de criterio.')
    print('Revisa los indices IDX en el Paso 5 o ejecuta CREST real.')


---
## 🌱 Semilla · Paso 7 — Figuras del ensemble

### 📈 Figura 1 — Perfil de energía y población acumulada

¿Cuántos confórmeros son suficientes para representar el 90% de la población?


In [ ]:
df_ens = pd.read_csv('p03_ensemble.csv').sort_values('dE_kJ_mol').reset_index(drop=True)
df_ens['pop_acum'] = df_ens['poblacion_pct'].cumsum()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

colores_bar = ['#0F2747' if d < 5 else '#2A6F97' if d < 12 else '#AABBD0'
               for d in df_ens['dE_kJ_mol']]
bars = ax1.bar(df_ens['conformero'], df_ens['dE_kJ_mol'],
               color=colores_bar, edgecolor='white', linewidth=0.4)
for bar, pct in zip(bars, df_ens['poblacion_pct']):
    if pct > 3:
        ax1.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3,
                 f'{pct:.0f}%', ha='center', va='bottom', fontsize=7)
ax1.set_xlabel('Conformero (por energia)', fontsize=11)
ax1.set_ylabel('dE / kJ/mol', fontsize=11)
ax1.set_title('Perfil energetico del ensemble', fontsize=11)

ax2.plot(range(1, len(df_ens)+1), df_ens['pop_acum'],
         'o-', color='#0F2747', ms=5, lw=2)
ax2.axhline(90, color='#888', ls='--', lw=1, label='90%')
ax2.axhline(99, color='#BBB', ls=':', lw=1, label='99%')
n_90 = int((df_ens['pop_acum'] < 90).sum()) + 1
ax2.axvline(n_90, color='#C03B26', ls='--', lw=1,
            label=f'{n_90} conf = 90%')
ax2.set_xlabel('Conformeros incluidos', fontsize=11)
ax2.set_ylabel('Poblacion acumulada (%)', fontsize=11)
ax2.set_title('Poblacion acumulada de Boltzmann', fontsize=11)
ax2.legend(fontsize=9); ax2.set_xlim(1, len(df_ens)); ax2.set_ylim(0,105)

plt.tight_layout()
plt.savefig('p03_perfil_energia.pdf', dpi=150, bbox_inches='tight')
plt.show()
print(f'Guardada: p03_perfil_energia.pdf')
print(f'Conformeros para el 90% de poblacion: {n_90}')


---
### 📈 Figura 2 — Mapa de Ramachandran del ensemble

Cada punto = un confórmero del ensemble. Color = población de Boltzmann.
Estrellas azules = mínimos de referencia MP2 (Guo et al. 2014).


In [ ]:
df_rama = df_conf.dropna(subset=['phi','psi']).copy()

fig, ax = plt.subplots(figsize=(6, 6))

sc = ax.scatter(df_rama['phi'], df_rama['psi'],
                c=df_rama['poblacion_pct'], cmap='YlOrRd',
                s=df_rama['poblacion_pct'] * 5 + 40,
                edgecolors='k', linewidths=0.5, zorder=5)

c1 = df_rama.sort_values('poblacion_pct', ascending=False).iloc[0]
ax.annotate(f'C1 ({c1.poblacion_pct:.0f}%)',
            xy=(c1.phi, c1.psi), xytext=(c1.phi+15, c1.psi+15),
            fontsize=8, arrowprops=dict(arrowstyle='->', lw=0.8))

for nombre, ref in referencia_rama.items():
    ax.scatter(ref['phi'], ref['psi'], marker='*', s=180,
               color='#2A6F97', zorder=6, edgecolors='k', lw=0.5)
    ax.text(ref['phi']-5, ref['psi']-12, nombre,
            fontsize=7, color='#2A6F97', ha='right')

ax.axhline(0, color='gray', lw=0.4, alpha=0.5)
ax.axvline(0, color='gray', lw=0.4, alpha=0.5)
ax.set_xlabel('phi (grados)', fontsize=12)
ax.set_ylabel('psi (grados)', fontsize=12)
ax.set_title(f'Mapa de Ramachandran -- {NOMBRE}\n'
             '(puntos: GFN2-xTB, estrellas: referencia MP2)',
             fontsize=11)
ax.set_xlim(-180, 180); ax.set_ylim(-180, 180)
fig.colorbar(sc, ax=ax, pad=0.02).set_label('Poblacion (%)', fontsize=9)
plt.tight_layout()
plt.savefig('p03_ramachandran.pdf', dpi=150, bbox_inches='tight')
plt.show()
print('Guardada: p03_ramachandran.pdf')


---
## 🌡️ Exploración — Poblaciones vs temperatura

Sin recalcular CREST, podemos explorar cómo evolucionan las poblaciones
conformacionales al variar la temperatura. Útil para:
- Espectroscopía a alta temperatura
- Comparar condiciones fisiológicas (310 K) vs criónicas (200 K)


In [ ]:
Ts = np.linspace(100, 700, 200)
dG = df_conf['dE_kJ_mol'].values
n_top = min(5, len(dG))

pop_T = np.zeros((len(Ts), n_top))
for k, T_i in enumerate(Ts):
    pesos_all = np.exp(-dG / (R * T_i))
    Z = pesos_all.sum()
    pop_T[k] = pesos_all[:n_top] / Z * 100

fig, ax = plt.subplots(figsize=(9, 5))
colors_t = ['#0F2747','#C03B26','#5A9BBF','#7BB37D','#E8A838']
for j in range(n_top):
    phi_j = df_conf.iloc[j]['phi']
    psi_j = df_conf.iloc[j]['psi']
    p298  = pop_T[abs(Ts-298).argmin(), j]
    ax.plot(Ts, pop_T[:,j], color=colors_t[j], lw=2,
            label=f'C{j+1} (phi={phi_j:.0f} psi={psi_j:.0f}) -- {p298:.0f}% @ 298K')

ax.axvline(298.15, color='gray', ls='--', lw=1, alpha=0.7)
ax.text(302, 3, '298 K', color='gray', fontsize=9)
ax.set_xlabel('Temperatura (K)', fontsize=12)
ax.set_ylabel('Poblacion (%)', fontsize=12)
ax.set_title(f'Top {n_top} conformeros vs temperatura', fontsize=11)
ax.legend(fontsize=8, loc='center right')
ax.set_xlim(100, 700); ax.set_ylim(0, 100)
plt.tight_layout()
plt.savefig('p03_pop_vs_T.pdf', bbox_inches='tight')
plt.show()
print('Guardada: p03_pop_vs_T.pdf')


---
## 🌳 El Bosque — 35 moléculas

| Clase | N | Ejemplos |
|---|---|---|
| Alcanos lineales C₄–C₁₀ | 7 | butano, pentano... decano |
| Dipéptidos modelo | 8 | Ac-Gly, **Ac-Ala** (semilla), Ac-Val, Ac-Phe... |
| Fármacos con flexibilidad conocida | 12 | ibuprofeno, paracetamol, omeprazol... |
| Macrociclos pequeños | 8 | ciclodecano, ciclododecano... |

**Pregunta clave:** ¿La relación entre enlaces rotativos y número de
confórmeros es lineal, exponencial, o depende de la clase estructural?


In [ ]:
BOSQUE_CSV = 'p03_bosque_resultados.csv'

if os.path.exists(BOSQUE_CSV):
    df_bosque = pd.read_csv(BOSQUE_CSV)
    print(f'Bosque cargado: {len(df_bosque)} moleculas')
    print(df_bosque['clase'].value_counts().to_dict())
else:
    print(f'No se encontro {BOSQUE_CSV} -- generando bosque sintetico...')
    np.random.seed(42)
    clases = ['alcano']*7 + ['dipeptido_modelo']*8 + ['farmaco']*12 + ['macrociclo']*8
    n_rot  = [2,3,4,5,6,7,8, 3,4,5,5,6,6,5,7, 3,2,4,6,7,5,4,8,3,6,5,4, 0,0,1,1,2,2,2,3]
    n_conf = [2,4,6,10,16,22,30, 3,8,12,15,18,20,14,25,
              5,3,8,18,22,12,9,28,6,20,14,10, 8,6,10,12,9,7,11,8]
    dE_C2  = np.random.uniform(1.5, 20, 35)
    pop1   = 100 / (1 + np.exp(-dE_C2 / (R * T_K)))
    ids    = ([f'alcano_C{i}' for i in range(4,11)] +
               ['Ac-Gly','Ac-Ala_bosque','Ac-Val','Ac-Phe',
                'Ac-Pro','Ac-Ser','Ac-Glu','Ac-Lys'] +
               [f'farmaco_{i:02d}' for i in range(1,13)] +
               [f'macrociclo_{i:02d}' for i in range(1,9)])
    df_bosque = pd.DataFrame({'id':ids,'clase':clases,'n_rot_bonds':n_rot,
                              'n_conformeros':n_conf,'dE_C2_kJ':dE_C2.round(2),
                              'pop_C1_pct':pop1.round(2)})
    df_bosque.to_csv(BOSQUE_CSV, index=False)
    print(f'Bosque sintetico generado: {len(df_bosque)} moleculas')

print(df_bosque.head())


In [ ]:
# Integrar la semilla calculada al dataset
semilla_row = pd.DataFrame([{
    'id'           : NOMBRE,
    'clase'        : 'dipeptido_modelo',
    'n_rot_bonds'  : 4,
    'n_conformeros': len(df_conf),
    'dE_C2_kJ'     : round(df_conf['dE_kJ_mol'].iloc[1], 2) if len(df_conf) > 1 else None,
    'pop_C1_pct'   : round(df_conf['poblacion_pct'].iloc[0], 2),
}])

df_final = pd.concat([df_bosque, semilla_row], ignore_index=True)
df_final.to_csv('p03_dataset_final.csv', index=False)
print(f'Dataset final: {len(df_final)} moleculas (bosque + semilla)')
print(df_final[df_final['id'] == NOMBRE])


---
## 📊 Análisis del bosque

### 📈 Figura 3 — Flexibilidad vs riqueza del ensemble

¿La relación entre enlaces rotativos y número de confórmeros es lineal o
exponencial? ¿Es igual en todas las clases estructurales?


In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
paleta = {'alcano':'#E74C3C','dipeptido_modelo':'#3498DB',
          'farmaco':'#27AE60','macrociclo':'#F39C12'}

for clase in df_final['clase'].unique():
    sub = df_final[df_final['clase']==clase].dropna(subset=['n_rot_bonds','n_conformeros'])
    ax.scatter(sub['n_rot_bonds'], sub['n_conformeros'],
               label=clase, color=paleta.get(clase,'#95A5A6'),
               alpha=0.85, s=60, edgecolors='k', lw=0.4)

sem = df_final[df_final['id']==NOMBRE]
if len(sem):
    ax.scatter(sem['n_rot_bonds'], sem['n_conformeros'],
               marker='*', s=280, color='gold', edgecolors='k',
               lw=0.8, zorder=6, label=f'Semilla ({NOMBRE})')

ax.set_xlabel('Numero de enlaces rotativos', fontsize=11)
ax.set_ylabel('Conformeros en ensemble CREST', fontsize=11)
ax.set_title('Flexibilidad molecular vs riqueza del ensemble', fontsize=12)
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig('p03_rotbonds_vs_nconf.pdf', dpi=150)
plt.show()
print('Guardada: p03_rotbonds_vs_nconf.pdf')
print()
print('=== Media por clase ===')
print(df_final.groupby('clase')[['n_rot_bonds','n_conformeros','pop_C1_pct']].mean().round(1))


---
### 📈 Figura 4 — Población del confórmero global por clase

¿En qué familias el confórmero global **no** representa la mayoría de la
población? Para esas moléculas es obligatorio calcular propiedades sobre el
ensemble completo, no sólo sobre el mínimo.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
clases_ord  = ['alcano','dipeptido_modelo','farmaco','macrociclo']
colores_cls = ['#E74C3C','#3498DB','#27AE60','#F39C12']

np.random.seed(0)
for j, (cls, col) in enumerate(zip(clases_ord, colores_cls)):
    sub = df_final[df_final['clase']==cls]['pop_C1_pct'].dropna()
    if not len(sub): continue
    jitter = np.random.uniform(-0.2, 0.2, len(sub))
    ax.scatter([j+ji for ji in jitter], sub, color=col,
               alpha=0.75, s=50, edgecolors='k', lw=0.4)
    ax.plot([j-0.3, j+0.3], [sub.mean()]*2, 'k-', lw=2.5, zorder=5)

ax.axhline(50, color='gray', ls='--', lw=0.8, label='50%')
ax.axhline(80, color='#555', ls=':', lw=0.8, label='80%')
ax.set_xticks(range(len(clases_ord)))
ax.set_xticklabels(clases_ord, rotation=15, fontsize=10)
ax.set_ylabel('Poblacion del conformero global p1 (%)', fontsize=11)
ax.set_title('Poblacion del conformero global por clase estructural', fontsize=12)
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('p03_pop_por_clase.pdf', dpi=150)
plt.show()
print('Guardada: p03_pop_por_clase.pdf')

baja = df_final[df_final['pop_C1_pct'] < 50]
print(f'\nMoleculas con p1 < 50% (ensemble obligatorio): {len(baja)}')
if len(baja):
    print(baja[['id','clase','n_rot_bonds','pop_C1_pct']].sort_values('pop_C1_pct').to_string(index=False))


---
## 🔄 Revisión de hipótesis iniciales

Compara tus respuestas previas con los resultados calculados.


In [ ]:
print('TUS RESPUESTAS INICIALES')
print('P1:', respuesta_1)
print('P2:', respuesta_2)
print('P3:', respuesta_3)
print()
print('RESULTADOS OBTENIDOS')
print(f'  Conformeros encontrados: {len(df_conf)}')
C1 = df_conf.iloc[0]
print(f'  Conformero dominante: C1 = {C1.poblacion_pct:.1f}%  phi={C1.phi:.0f} psi={C1.psi:.0f}')
p_anti = 100 / (1 + 2*np.exp(-2.5/(R*T_K)))
print(f'  Fraccion anti butano (teorica, 298 K): {p_anti:.1f}%')


In [ ]:
reflexion = (
    '1. Corresponde el conformero dominante con el C7eq predicho?\n'
    '\n'
    '2. Lo mas sorprendente del ensemble del bosque fue...\n'
    '\n'
    '3. Para un calculo de docking con un farmaco de 5-8 rot_bonds,\n'
    '   usar solo el minimo seria un problema porque...\n'
    '\n'
    '4. Una pregunta nueva que me genera esta practica:...'
)
print(reflexion)


---
## 📦 Entregables

| Tipo | Archivo | Descripción |
|---|---|---|
| **D1** | `crest_conformers.xyz` | Ensemble completo de la semilla |
| **D2** | `p03_ensemble.csv` | ΔE, poblaciones, φ, ψ de cada confórmero |
| **D3** | `p03_dataset_final.csv` | Bosque completo + semilla |
| **D4** | `crest_run.log` | Log de CREST sin modificar |
| **F1** | `p03_perfil_energia.pdf` | Perfil energético + población acumulada |
| **F2** | `p03_ramachandran.pdf` | Mapa de Ramachandran coloreado por población |
| **F3** | `p03_pop_vs_T.pdf` | Población vs temperatura (100–700 K) |
| **F4** | `p03_rotbonds_vs_nconf.pdf` | Flexibilidad vs riqueza del ensemble |
| **F5** | `p03_pop_por_clase.pdf` | Población p₁ por clase estructural |
| **T1** | Reporte ≤ 2 págs. | Validación Ramachandran + F1 + F4 comentadas |
| **T2** | Respuestas discusión | ≤ 150 palabras por pregunta |


In [ ]:
from pathlib import Path
entregables = [
    ('D1','crest_conformers.xyz'), ('D2','p03_ensemble.csv'),
    ('D3','p03_dataset_final.csv'), ('D4','crest_run.log'),
    ('F1','p03_perfil_energia.pdf'), ('F2','p03_ramachandran.pdf'),
    ('F3','p03_pop_vs_T.pdf'), ('F4','p03_rotbonds_vs_nconf.pdf'),
    ('F5','p03_pop_por_clase.pdf'),
]
ok_count = 0
for tipo, archivo in entregables:
    p = Path(archivo)
    ok = p.exists() and p.stat().st_size > 0
    print(f'  [{"OK" if ok else "FALTA"}] {tipo}: {archivo}')
    if ok: ok_count += 1
print(f'\n{ok_count}/{len(entregables)} entregables listos.')


---
## ❓ Preguntas de discusión

Responde en el informe T2 (≤ 150 palabras por pregunta).


In [ ]:
preguntas = {
    1: '(Comprension) Cuantas rondas de iMTD-GC ejecuto CREST? '
       'Convergieron el numero de conformeros entre la penultima y ultima ronda?',
    2: '(Comprension) Diferencia de energia entre C1 y C2 en kJ/mol '
       'y en unidades de kBT a 298 K.',
    3: '(Analisis) El C7eq domina en vacio; en agua domina C5. '
       'Que fuerza intermolecular explica el cambio? '
       'Como modificarias el protocolo para capturarlo?',
    4: '(Analisis) Molecula del bosque con MAS conformeros y con MENOS. '
       'Corresponde al mayor/menor numero de enlaces rotativos? Si no, por que?',
    5: '(Metodologico) Efecto de cambiar rthr de 0.125 a 0.5 A vs 0.01 A '
       'en el tamano del ensemble.',
    6: '(Metodologico) En que clase de moleculas el orden de conformeros '
       'podria cambiar significativamente al recalcular con DFT/def2-TZVP?',
}
for n, t in preguntas.items():
    print(f'P{n}. {t}\n')


---
## 🔗 Conexión con el resto del manual

| Habilidad | Se usa en... |
|---|---|
| Búsqueda conformacional CREST | Bloque 2+: `crest_best.xyz` input estándar para DFT |
| Ensemble + Boltzmann | P42: promedio conformacional en trayectorias DM |
| Mapa de Ramachandran | Prácticas de péptidos del Bloque 4 |
| Población vs temperatura | P17, P28: equilibrios en distintos solventes |
| Dataset conformacional | P43 (docking): usar ensemble completo |

**Archivos para usar en prácticas futuras:**
- `crest_best.xyz` → input para **P4** (DFT/B3LYP)
- `crest_conformers.xyz` → ensemble para **P43** (docking)
- `p03_ensemble.csv` → referencia para **P41** (dinámica molecular)
- `p03_dataset_final.csv` → análisis de espacio conformacional

---

> ✏️ **Nota al manual — recursos a añadir en el LaTeX:**
>
> 1. **Diagrama de flujo iMTD-GC** en el marco teórico.
> 2. **Mapa de Ramachandran de referencia** (MP2/6-311+G**) para comparación directa.
> 3. **Tabla comparativa de métodos de búsqueda conformacional:** CREST vs
>    sistemática vs Monte Carlo vs DM clásica.
> 4. **Párrafo sobre el dataset GEOM** (37M confórmeros, doi:10.1038/s41597-022-01288-4).
